In [12]:
import pandas as pd
import xgboost as xgb
from sklearn.model_selection import train_test_split

# 1. Load dataset
df = pd.read_csv('../data/raw/cleaned_credit_risk_dataset.csv')

# 2. Robust target mapping (handles extra spaces and letter casing)
if 'Decision' in df.columns:
    df['Decision'] = (
        df['Decision']
        .astype(str)
        .str.strip()
        .str.upper()
        .map({'APPROVE': 1, 'APPROVED': 1, '1': 1, 'REJECT': 0, 'REJECTED': 0, '0': 0})
    )

# 3. Feature Engineering matching exact CSV column headers
if 'Loan Amount' in df.columns and 'Annual Income' in df.columns:
    df['Debt_to_Income_Ratio'] = df['Loan Amount'] / (df['Annual Income'] + 1)

if 'Annual Income' in df.columns and 'Number of Dependents' in df.columns:
    df['Income_per_Dependent'] = df['Annual Income'] / (df['Number of Dependents'] + 1)

# 4. Prepare X and y
X = df.drop(columns=['Decision'])
y = df['Decision'].astype(int)

# Automatically convert any remaining string features to numeric dummies
X = pd.get_dummies(X, drop_first=True)

# 5. Calculate class imbalance weight safely
pos_count = (y == 1).sum()
neg_count = (y == 0).sum()
ratio = neg_count / pos_count if pos_count > 0 else 1.0

# 6. Stratified train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 7. Fit tuned XGBoost model
model = xgb.XGBClassifier(
    n_estimators=200,
    max_depth=4,
    learning_rate=0.05,
    scale_pos_weight=ratio,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric='logloss',
    random_state=42
)
model.fit(X_train, y_train)

# 8. Save trained model
model.save_model('../models/xgboost_credit_model.json')
print("Model trained and saved successfully to ../models/xgboost_credit_model.json!")

Model trained and saved successfully to ../models/xgboost_credit_model.json!
